In [0]:
#| default_exp core

## Module setup

This notebook is the source of truth for the `nbskill.core` module. It starts with imports and shared helper functions used by the CLI commands below.

In [ ]:
#| export
import ast
import copy
import hashlib
import json
import os
import re
import sys
import time
from contextlib import contextmanager
from functools import wraps
from importlib.resources import files
from pathlib import Path
from chkstyle.core import main as _chkstyle_main
from execnb.shell import CaptureShell
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import Param, call_parse, _in_call_parse
from fastcore.xtras import pglob
from nbdev.diff import nbs_pair, source_diff
from nbdev.doclinks import nbdev_export

## Shared notebook mechanics

These helpers parse CLI values, normalize notebook cells, validate generated Python, and keep edits safe before anything is written.

In [0]:
#| export
def _cli_return(value=None):
    return None if _in_call_parse.get() else value

In [0]:
#| export
def _cli_error(msg):
    if _in_call_parse.get():
        print(msg, file=sys.stderr)
        raise SystemExit(1)
    raise ValueError(msg)

In [0]:
#| export
def _failure_map_path():
    return Path(os.environ.get("NBSKILL_FAILURE_MAP", ".nbskill-errors.json")).expanduser()

In [0]:
#| export
def _empty_failure_map():
    return {"version": 1, "events": [], "counts": {}, "last_call": None}

In [0]:
#| export
def _load_failure_map(path):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        data = _empty_failure_map()
    data.setdefault("version", 1)
    data.setdefault("events", [])
    data.setdefault("counts", {})
    data.setdefault("last_call", None)
    return data

In [0]:
#| export
def _bump_count(data, kind, tool):
    counts = data.setdefault("counts", {})
    group = counts.setdefault(kind, {})
    group[tool] = group.get(tool, 0) + 1

In [0]:
#| export
def _write_failure_map(path, data):
    data["events"] = data.get("events", [])[-200:]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

In [0]:
#| export
def _record_tool_start(tool):
    path = _failure_map_path()
    now = time.time()
    event = {"tool": tool, "ts": now}
    try:
        data = _load_failure_map(path)
        last = data.get("last_call")
        if last:
            delta = now - float(last.get("ts", now))
            reasons = []
            if last.get("tool") == tool: reasons.append("same_tool")
            if delta <= 1.0: reasons.append("within_1s")
            if reasons:
                _bump_count(data, "friction", tool)
                data["events"].append({
                    "kind": "friction",
                    "tool": tool,
                    "previous_tool": last.get("tool"),
                    "seconds_since_previous": round(delta, 3),
                    "reasons": reasons,
                    "ts": now,
                })
        data["last_call"] = event
        _write_failure_map(path, data)
    except OSError:
        pass
    return event

In [0]:
#| export
def _record_tool_failure(event, exc):
    path = _failure_map_path()
    try:
        data = _load_failure_map(path)
        tool = event["tool"]
        _bump_count(data, "failures", tool)
        data["events"].append({
            "kind": "failure",
            "tool": tool,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "ts": time.time(),
        })
        _write_failure_map(path, data)
    except OSError:
        pass

In [0]:
#| export
@contextmanager
def _track_tool(tool):
    event = _record_tool_start(tool)
    try:
        yield
    except BaseException as exc:
        _record_tool_failure(event, exc)
        raise

In [0]:
#| export
def _tracked_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with _track_tool(func.__name__):
            return func(*args, **kwargs)
    return wrapper

In [0]:
#| export
def _parse_literal(value):
    if value is None: return None
    if isinstance(value, str):
        value = value.strip()
        if value.lower() in {"", "none", "null"}: return None
        try: return ast.literal_eval(value)
        except (SyntaxError, ValueError): return value
    return value

In [0]:
#| export
def _none_if_string(value):
    return None if isinstance(value, str) and value.strip().lower() in {"", "none", "null"} else value

In [0]:
#| export
def _parse_slice(value):
    if not isinstance(value, str) or ":" not in value: return None
    parts = value.split(":")
    if len(parts) not in (2, 3): return None
    vals = [int(p) if p else None for p in parts]
    return slice(*vals)

In [0]:
#| export
def _as_index(value, length):
    idx = int(value)
    if idx < 0: idx += length
    if idx < 0 or idx >= length: raise IndexError(value)
    return idx

In [0]:
#| export
def _parse_read_selector(value):
    value = _parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)): return [int(o) for o in value]
    return int(value)

In [0]:
#| export
def _parse_write_target(value):
    value = _parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)):
        if len(value) != 2: raise ValueError("write ranges must have start and stop")
        return slice(value[0], value[1])
    return int(value)

In [0]:
#| export
def _select_cells(cells, selector):
    items = list(enumerate(cells))
    if selector is None: return items
    selector = _parse_read_selector(selector)
    if isinstance(selector, slice): return items[selector]
    if isinstance(selector, list):
        return [(idx, cells[idx]) for idx in (_as_index(o, len(cells)) for o in selector)]
    idx = _as_index(selector, len(cells))
    return [(idx, cells[idx])]

In [0]:
#| export
def _delete_cells(cells, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    if isinstance(target, slice):
        del cells[target]
        return
    idx = _as_index(target, len(cells))
    del cells[idx]

In [0]:
#| export
def _split_blocks(text):
    text = "" if text is None else str(text)
    if not text: return []
    return [o.strip("\n") for o in re.split(r"(?m)^\s*---\s*$", text) if o.strip()]

In [0]:
#| export
def _coerce_cell(cell, default_type="code"):
    if isinstance(cell, dict): return cell
    if isinstance(cell, (tuple, list)) and len(cell) == 2:
        cell_type, source = cell
        return mk_cell(str(source), cell_type=str(cell_type))
    return mk_cell(str(cell), cell_type=default_type)

In [0]:
#| export
def _is_definition_node(node):
    return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))

In [0]:
#| export
def _node_start_line(node):
    return min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1

In [0]:
#| export
def _is_export_directive(line):
    return re.match(r"^\s*#\|\s*(export|exports|exporti)(\s|$)", line) is not None

In [0]:
#| export
def _is_export_gap(lines):
    return bool(lines) and all((not line.strip()) or _is_export_directive(line) for line in lines)

In [0]:
#| export
def _emit_code_chunk(chunks, lines, export_prefix=None):
    if export_prefix: lines = [*export_prefix, *lines]
    text = "\n".join(lines).strip("\n")
    if text: chunks.append(text)

In [0]:
#| export
def _split_code_cell_sources(source):
    source = source.strip("\n")
    if not source: return []
    try: tree = ast.parse(source)
    except SyntaxError: return [source]
    if sum(1 for node in tree.body if _is_definition_node(node)) <= 1: return [source]

    lines = source.splitlines()
    first_start = _node_start_line(tree.body[0]) if tree.body else 0
    leading = lines[:first_start]
    shared_export = [line for line in leading if _is_export_directive(line)] if _is_export_gap(leading) else []
    chunks = []
    if shared_export:
        cursor = first_start
    else:
        _emit_code_chunk(chunks, leading)
        cursor = first_start

    for node in tree.body:
        start = _node_start_line(node)
        end = node.end_lineno
        gap = lines[cursor:start]
        if shared_export and _is_export_gap(gap): gap = []
        _emit_code_chunk(chunks, [*gap, *lines[start:end]], shared_export or None)
        cursor = end
    _emit_code_chunk(chunks, lines[cursor:])
    return chunks or [source]

In [0]:
#| export
def _split_code_cell(cell):
    cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
    if cell_type != "code": return [cell]
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): source = "".join(source)
    sources = _split_code_cell_sources(str(source))
    if len(sources) <= 1: return [cell]
    return [mk_cell(source, cell_type="code") for source in sources]

In [0]:
#| export
def _split_symbol_cells(cells):
    split = []
    for cell in cells: split.extend(_split_code_cell(cell))
    return split

In [0]:
#| export
def _cell_source(cell):
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): return "".join(source)
    return str(source)

In [0]:
#| export
def _cell_hash(cell_or_source, n=12):
    source = _cell_source(cell_or_source) if not isinstance(cell_or_source, str) else cell_or_source
    digest = hashlib.sha256(source.encode("utf-8")).hexdigest()
    return digest if n is None else digest[:n]

In [0]:
#| export
def _parse_one_cell(text, default_type="code"):
    blocks = _split_blocks(text)
    if len(blocks) != 1: _cli_error("update_cell expects exactly one replacement cell")
    return _coerce_cell(blocks[0], default_type)

In [0]:
#| export
def _cell_matches_hash(cell, source_hash):
    if source_hash is None: return True
    return _cell_hash(cell, n=None).startswith(str(source_hash).lower())

In [0]:
#| export
def _find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == cell_id]
    if len(matches) == 1: return matches[0]
    if not matches: _cli_error(f"No cell has id {cell_id!r}")
    _cli_error(f"Multiple cells have id {cell_id!r}")

In [0]:
#| export
def _find_cell_by_text(cells, old_str):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if old_str in _cell_source(cell)]
    if len(matches) == 1: return matches[0]
    if not matches: _cli_error("old_str did not match any cell")
    idxs = ", ".join(str(idx) for idx, _ in matches)
    _cli_error(f"old_str matched multiple cells: {idxs}. Use --cell_id or a more specific old_str.")

In [0]:
#| export
def _replace_cell(nb, idx, new_cell):
    old_id = getattr(nb.cells[idx], "id", None)
    if old_id is not None: new_cell.id = old_id
    nb.cells[idx] = new_cell

In [0]:
#| export
def _clear_outputs(cell):
    if getattr(cell, "cell_type", None) == "code":
        cell.outputs = []
        cell.execution_count = None
    return cell

In [0]:
#| export
def _load_cells_text(cells="", cells_file=None):
    if cells_file:
        if cells: raise ValueError("Use either cells or cells_file, not both")
        return Path(cells_file).expanduser().read_text(encoding="utf-8")
    if cells == "-": return sys.stdin.read()
    return cells

In [0]:
#| export
def _should_validate_python(source):
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("%", "!")): return False
    return bool(source.strip())

In [0]:
#| export
def _format_syntax_error(source, err, cell_idx):
    lines = source.splitlines()
    line = lines[err.lineno - 1] if err.lineno and 0 < err.lineno <= len(lines) else ""
    pointer = " " * max((err.offset or 1) - 1, 0) + "^" if line else ""
    msg = [f"Invalid Python in new code cell {cell_idx}: {err.msg} at line {err.lineno}, column {err.offset}"]
    if line: msg += [line, pointer]
    msg.append("Tip: shell quoting can turn backslash-n escapes into real newlines inside Python strings. Use --cells_file PATH or cells=- for complex code.")
    return chr(10).join(msg)

In [0]:
#| export
def _validate_code_cells(cells):
    for idx, cell in enumerate(cells):
        cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
        if cell_type != "code": continue
        source = _cell_source(cell)
        if not _should_validate_python(source): continue
        try: ast.parse(source)
        except SyntaxError as err:
            msg = _format_syntax_error(source, err, idx)
            if _in_call_parse.get(): raise SystemExit(msg)
            raise ValueError(msg) from err

In [0]:
#| export
def _parse_cells(cells, default_type="code"):
    if isinstance(cells, (list, tuple)): return _split_symbol_cells([_coerce_cell(o, default_type) for o in cells])

    parsed = []
    for block in _split_blocks(cells):
        lines = block.splitlines()
        marker = lines[0].strip().lower() if lines else ""
        cell_type = default_type
        if marker in {"%%markdown", "%%md"}:
            cell_type, lines = "markdown", lines[1:]
        elif marker == "%%code":
            cell_type, lines = "code", lines[1:]
        elif marker == "%%raw":
            cell_type, lines = "raw", lines[1:]
        parsed.append(mk_cell("\n".join(lines), cell_type=cell_type))
    return _split_symbol_cells(parsed)

In [0]:
#| export
def _first_line(source):
    for line in source.splitlines():
        line = line.strip()
        if line: return line
    return ""

In [0]:
#| export
def _cell_prefix(idx, cell, show_ids=False):
    suffix = f" hash={_cell_hash(cell)}" if show_ids else ""
    return f"Cell id={cell.id}{suffix}: {cell.cell_type}"

In [0]:
#| export
def _format_chapter_spans(spans, cells, show_ids=False):
    lines = []
    for span in spans:
        cell = cells[span["start"]]
        suffix = f" hash={_cell_hash(cell)}" if show_ids else ""
        lines.append(f"Chapter id={cell.id}{suffix}: ## {span['title']}")
    return "\n".join(lines)

In [0]:
#| export
def _format_overview(items, show_ids=False):
    lines = []
    for idx, cell in items:
        summary = _first_line(cell.source)
        lines.append(f"{_cell_prefix(idx, cell, show_ids)} | {summary}")
    return "\n".join(lines)

In [0]:
#| export
def _markdown_overview(cell):
    lines = cell.source.splitlines()
    if not any(re.match(r"^#{1,2}\s+", line.strip()) for line in lines): return []
    text = cell.source.strip()
    return [text] if text else []


def _definition_lines(node, indent=""):
    tmp = copy.deepcopy(node)
    tmp.body = [ast.Pass()]
    ast.fix_missing_locations(tmp)
    lines = []
    for line in ast.unparse(tmp).splitlines():
        if line.strip() == "pass": continue
        lines.append(f"{indent}{line}" if line else line)
    return lines


def _docstring_lines(node, indent="    "):
    doc = ast.get_docstring(node)
    if not doc: return []
    quote = indent + chr(34) * 3
    return [quote, *[f"{indent}{line}" for line in doc.splitlines()], quote]


def _function_overview(node, indent=""):
    return [*_definition_lines(node, indent=indent), *_docstring_lines(node, indent=indent + "    ")]


def _class_overview(node):
    lines = [*_definition_lines(node), *_docstring_lines(node)]
    init = next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == "__init__"), None)
    if init:
        if lines: lines.append("")
        lines += _function_overview(init, indent="    ")
    return lines


def _code_overview(cell):
    try: tree = ast.parse(cell.source)
    except SyntaxError: return []
    lines = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): lines += _function_overview(node)
        elif isinstance(node, ast.ClassDef): lines += _class_overview(node)
        if lines and lines[-1] != "": lines.append("")
    if lines and lines[-1] == "": lines.pop()
    return lines


def _format_headers(items, show_ids=False):
    chunks = []
    for idx, cell in items:
        if cell.cell_type == "markdown": lines = _markdown_overview(cell)
        elif cell.cell_type == "code": lines = _code_overview(cell)
        else: lines = []
        if lines: chunks.append(f"{_cell_prefix(idx, cell, show_ids)}\n" + "\n".join(lines))
    return "\n\n".join(chunks)

In [0]:
#| export
def _format_full(items, show_ids=False):
    chunks = []
    for idx, cell in items:
        chunks.append(f"{_cell_prefix(idx, cell, show_ids)}\n{cell.source}")
    return "\n\n".join(chunks)

In [0]:
#| export
def _matches_filter(source, pattern):
    pattern = str(pattern)
    if pattern in source: return True
    try: return re.search(pattern, source, flags=re.MULTILINE) is not None
    except re.error: return False

In [0]:
#| export
def _format_filter(items, show_ids=False):
    return "\n\n".join(f"{_cell_prefix(idx, cell, show_ids)}\n{cell.source}" for idx, cell in items)

In [ ]:
#| export
def _is_exported_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    return any(_is_export_directive(line) for line in _cell_source(cell).splitlines())


def _normalize_cell_type_filter(value):
    if value is None: return None
    aliases = {
        "code": "code",
        "py": "code",
        "python": "code",
        "md": "markdown",
        "markdown": "markdown",
        "doc": "markdown",
        "docs": "markdown",
        "raw": "raw",
        "export": "export",
        "exported": "export",
    }
    normalized = set()
    for item in str(value).split(","):
        key = item.strip().lower()
        if not key: continue
        if key not in aliases: raise ValueError(f"Unknown cell_type {item!r}; use code, md, raw, or export")
        normalized.add(aliases[key])
    return normalized or None


def _cell_matches_type(cell, cell_type):
    wanted = _normalize_cell_type_filter(cell_type)
    if wanted is None: return True
    if "export" in wanted and _is_exported_code_cell(cell): return True
    return getattr(cell, "cell_type", None) in (wanted - {"export"})

In [0]:
#| export
def _with_context(cells, items, context=0):
    context = int(context or 0)
    if context < 0: raise ValueError("context must be >= 0")
    if context == 0: return items

    idxs = {idx for idx, _ in items}
    for idx in list(idxs):
        for offset in range(1, context + 1):
            prev = idx - offset
            if prev < 0: break
            if cells[prev].cell_type == "markdown": idxs.add(prev)
        for offset in range(1, context + 1):
            nxt = idx + offset
            if nxt >= len(cells): break
            if cells[nxt].cell_type == "code" and not _is_exported_code_cell(cells[nxt]): idxs.add(nxt)
    return [(idx, cells[idx]) for idx in sorted(idxs)]

In [0]:
#| export
def _chapter_title(cell):
    if getattr(cell, "cell_type", None) != "markdown": return None
    for line in _cell_source(cell).splitlines():
        match = re.match(r"^##\s+(.+?)\s*$", line.strip())
        if match: return match.group(1).strip()
    return None

In [0]:
#| export
def _chapter_spans(cells):
    starts = [(idx, title) for idx, cell in enumerate(cells) if (title := _chapter_title(cell))]
    spans = []
    for pos, (start, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        spans.append(dict(title=title, start=start, end=end))
    return spans

In [0]:
#| export
def _matching_chapters(cells, chapter=None):
    spans = _chapter_spans(cells)
    if chapter is None: return spans
    return [span for span in spans if _matches_filter(span["title"], chapter)]

In [0]:
#| export
def _chapter_index_set(cells, chapter):
    idxs = set()
    for span in _matching_chapters(cells, chapter):
        idxs.update(range(span["start"], span["end"]))
    return idxs

In [0]:
#| export
def _one_chapter(cells, chapter, create=False):
    matches = _matching_chapters(cells, chapter)
    if len(matches) == 1: return matches[0]
    if not matches and create:
        cells.append(mk_cell(f"## {chapter}", cell_type="markdown"))
        return dict(title=str(chapter), start=len(cells) - 1, end=len(cells))
    if not matches: raise ValueError(f"No chapter matches {chapter!r}")
    titles = ", ".join(f"{span['title']} ({span['start']}:{span['end']})" for span in matches)
    raise ValueError(f"Chapter {chapter!r} matches multiple chapters: {titles}")

In [0]:
#| export
def _chapter_body_len(span):
    return max(span["end"] - span["start"] - 1, 0)

In [0]:
#| export
def _chapter_body_slice(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    start, stop, step = target.indices(body_len)
    if step != 1: raise ValueError("chapter ranges do not support steps")
    return slice(body_start + start, body_start + stop)

In [0]:
#| export
def _chapter_delete(cells, span, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if isinstance(target, slice):
        del cells[_chapter_body_slice(span, target)]
        return
    idx = int(target)
    if idx < 0: idx += body_len
    if idx < 0 or idx >= body_len: raise IndexError(target)
    del cells[body_start + idx]

In [0]:
#| export
def _chapter_insert_target(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if target is None: return slice(body_start, body_start + body_len)
    if isinstance(target, slice): return _chapter_body_slice(span, target)
    idx = int(target)
    if idx == -1: return body_start + body_len
    if idx < 0: idx += body_len
    if idx < 0 or idx > body_len: raise IndexError(target)
    return body_start + idx

## Reading notebooks

The read path presents notebooks as compact, non-JSON text. It favors stable cell IDs over shifting cell numbers and can include nearby documentation or experiment context.

In [ ]:
#| export
@call_parse
@_tracked_call
def read_nb(
    path: str,  # Notebook path
    cell_id: str | None = None,  # Stable notebook cell id to select
    chapter: str | None = None,  # Chapter title string or regex to select
    cell_type: str | None = None,  # Filter cells: code/py, md/markdown, raw, export; comma-separated is allowed
    contains: str | None = None,  # Include only cells whose source contains this text
    filter: str | None = None,  # Regex or string; print only matching cell ids and sources
    context: int = 0,  # Add up to N previous markdown and following non-export code cells
    show_ids: bool = False,  # Include source hashes in output
    scope: Param("overview, outline, or full", str, choices=("overview", "outline", "full")) = "overview",
):
    "Print a compact, non-JSON view of a notebook."
    if context < 0: raise ValueError("context must be >= 0")

    nb = _read_nb(path)
    items = [_find_cell_by_id(nb.cells, cell_id)] if cell_id is not None else list(enumerate(nb.cells))
    if chapter is not None:
        chapter_idxs = _chapter_index_set(nb.cells, chapter)
        items = [(i, c) for i, c in items if i in chapter_idxs]
    if cell_type is not None: items = [(i, c) for i, c in items if _cell_matches_type(c, cell_type)]
    if contains is not None: items = [(i, c) for i, c in items if contains in c.source]
    if filter is not None: items = [(i, c) for i, c in items if _matches_filter(c.source, filter)]
    items = _with_context(nb.cells, items, context)

    if filter is not None: text = _format_filter(items, show_ids=show_ids)
    elif scope == "overview": text = _format_overview(items, show_ids=show_ids)
    elif scope == "outline": text = _format_headers(items, show_ids=show_ids)
    else: text = _format_full(items, show_ids=show_ids)
    if text: print(text)
    return _cli_return(text)

In [0]:
#| export
def _chstyle_argv(path=".", skip_folder_re=None, skip_path=None):
    path = "." if path is None else str(path)
    argv = ["chstyle", path]
    if skip_folder_re: argv += ["--skip-folder-re", str(skip_folder_re)]
    if skip_path: argv += ["--skip-path", str(skip_path)]
    return argv

In [0]:
#| export
def _run_chstyle(path=".", skip_folder_re=None, skip_path=None, strict=False):
    status = _chkstyle_main(_chstyle_argv(path, skip_folder_re, skip_path))
    if strict and status: raise SystemExit(status)
    return status

In [ ]:
#| export
@call_parse
@_tracked_call
def chstyle(
    path: Param("File or folder to check", str, opt=False, nargs="?") = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Folder name/path to skip
    strict: bool = False,  # Exit non-zero when style hints are found
):
    "Print fast.ai style hints using fastaistyle/chkstyle."
    status = _run_chstyle(path, skip_folder_re, skip_path, strict)
    return _cli_return(status)

## Writing notebooks

The write path appends, inserts, replaces, and validates cells while keeping nbdev exports in sync. ID-based insertion is preferred when the target must survive cell movement.

In [ ]:
#| export
@call_parse
@_tracked_call
def write_nb(
    path: str,  # Notebook path
    cells: Param("Cell block text", str, opt=False, nargs="?") = "",  # Cells to write; use - to read stdin
    cells_file: str | None = None,  # Read cell block text from a UTF-8 file to avoid shell escaping
    before_id: str | None = None,  # Insert before this stable cell id
    after_id: str | None = None,  # Insert after this stable cell id
    chapter: str | None = None,  # Chapter title string or regex; missing chapters are created
    replace: bool = False,  # Replace the full notebook, or the selected chapter body
    cell_type: str = "code",  # Default type for cells without %% marker
    export: bool = True,  # Run nbdev-export after writing
    run_test: bool = False,  # Execute the notebook with execnb after writing
    run_style: bool = False,  # Run chstyle after writing
    style_strict: bool = False,  # Fail when chstyle finds hints
    validate_code: bool = True,  # Validate new Python code cells before writing
):
    "Write cells to a notebook using append, replace, id anchors, or chapters."
    if before_id and after_id: _cli_error("Use either before_id or after_id, not both")
    if (before_id or after_id) and chapter is not None: _cli_error("Use id-based insertion or chapter insertion, not both")
    if (before_id or after_id) and replace: _cli_error("Use id-based insertion or replace, not both")
    path = Path(path)
    cells = _load_cells_text(cells, cells_file)
    new_cells = _parse_cells(cells, cell_type)
    if validate_code: _validate_code_cells(new_cells)

    if replace and chapter is None:
        nb = new_nb(new_cells)
    else:
        nb = _read_nb(path) if path.exists() else new_nb([])
        if chapter is not None:
            span = _one_chapter(nb.cells, chapter, create=True)
            if replace:
                del nb.cells[span["start"] + 1:span["end"]]
                target = span["start"] + 1
            else:
                target = span["end"]
        elif before_id or after_id:
            idx, _ = _find_cell_by_id(nb.cells, before_id or after_id)
            target = idx if before_id else idx + 1
        else:
            target = len(nb.cells)
        for offset, cell in enumerate(new_cells):
            nb.cells.insert(target + offset, cell)

    _write_nb(nb, path)
    if export: nbdev_export(path=str(path))
    msg = f"Wrote {len(nb.cells)} cells to {path}"
    if replace: msg += " using replace"
    if chapter is not None: msg += f" in chapter {chapter!r}"
    if before_id: msg += f" before id={before_id}"
    if after_id: msg += f" after id={after_id}"
    if export: msg += " and exported with nbdev"
    print(msg)
    if run_test: _run_notebook_test(path)
    if run_style:
        print(f"Running chstyle on {path}")
        _run_chstyle(path, strict=style_strict)
    return _cli_return(path)

In [0]:
#| export
def _save_nb(nb, path, export=True):
    _write_nb(nb, path)
    if export: nbdev_export(path=str(path))

## Executing notebooks

Execution uses `execnb` directly so command-line output and failures remain visible, even in restricted environments where nbdev's worker checks may fail.

In [0]:
#| export
def _exec_limiters(up2id):
    up2id = _parse_literal(up2id)
    noop = lambda cell: None
    if up2id is None: return (lambda cell: False), noop
    if isinstance(up2id, int):
        if up2id < 0: raise ValueError("up2id must be >= 0")
        return (lambda cell: cell.idx_ >= up2id), noop

    done = False
    def preproc(cell): return done
    def postproc(cell):
        nonlocal done
        if cell.id == str(up2id): done = True
    return preproc, postproc

In [ ]:
#| export
def _text_output(value):
    if value is None: return ""
    if isinstance(value, list): return "".join(map(str, value))
    return str(value)


def _output_text(output):
    otype = output.get("output_type")
    if otype == "stream": return _text_output(output.get("text"))
    if otype == "error":
        tb = output.get("traceback")
        if tb: return _text_output(tb)
        return f"{output.get('ename', 'Error')}: {output.get('evalue', '')}"
    if otype in {"execute_result", "display_data"}:
        data = output.get("data", {})
        for mime in ("text/plain", "text/markdown", "text/html"):
            if mime in data: return _text_output(data[mime])
    return ""


def _is_rich_traceback_stream(output):
    if output.get("output_type") != "stream": return False
    text = _text_output(output.get("text"))
    return "Traceback" in text and "\x1b[" in text


def _executed_cells(nb, up2id=None):
    up2id = _parse_literal(up2id)
    if up2id is None: return list(enumerate(nb.cells))
    if isinstance(up2id, int): return list(enumerate(nb.cells[:up2id]))
    items = []
    for idx, cell in enumerate(nb.cells):
        items.append((idx, cell))
        if cell.id == str(up2id): break
    return items


def _print_nb_outputs(path, up2id=None):
    nb = _read_nb(path)
    for idx, cell in _executed_cells(nb, up2id):
        outputs = getattr(cell, "outputs", None) or []
        has_error = any(output.get("output_type") == "error" for output in outputs)
        for output in outputs:
            if has_error and _is_rich_traceback_stream(output): continue
            text = _output_text(output)
            if not text: continue
            print(f"--- output id={cell.id} ---")
            print(text, end="" if text.endswith("\n") else "\n")

In [ ]:
#| export
@call_parse
@_tracked_call
def exec_nb(
    path: str,  # Notebook path
    dest: str | None = None,  # Destination path; defaults to overwriting path
    exc_stop: bool = False,  # Stop on exceptions
    up2id: int | str | None = None,  # Execute first N cells, or through this cell id
    chapter: str | None = None,  # Execute through this chapter, inclusive
    show_output: bool = True,  # Print saved cell outputs and errors after execution
    verbose: bool = False,  # Show stdout/stderr live while executing
):
    "Execute a notebook with execnb and save outputs."
    dest = dest or path
    chapter_title = None
    if chapter is not None:
        if up2id is not None: raise ValueError("Use either chapter or up2id, not both")
        nb = _read_nb(path)
        span = _one_chapter(nb.cells, chapter)
        up2id, chapter_title = span["end"], span["title"]
    preproc, postproc = _exec_limiters(up2id)
    CaptureShell().execute(path, dest=dest, exc_stop=exc_stop, preproc=preproc, postproc=postproc, verbose=verbose)
    msg = f"Executed {path} -> {dest}"
    if chapter_title is not None: msg += f" (chapter={chapter_title!r}, up2id={up2id})"
    elif up2id is not None: msg += f" (up2id={up2id})"
    print(msg)
    if show_output: _print_nb_outputs(dest, up2id=up2id)
    return _cli_return(Path(dest))

In [0]:
#| export
def _notebook_error_summaries(path, up2id=None):
    nb = _read_nb(path)
    errors = []
    for idx, cell in _executed_cells(nb, up2id=up2id):
        for output in cell.get("outputs", []):
            if output.get("output_type") == "error":
                ename = output.get("ename", "Error")
                evalue = output.get("evalue", "")
                errors.append(f"id={cell.id} {ename}: {evalue}".strip())
    return errors

In [ ]:
#| export
def _run_notebook_test(path):
    print(f"Running notebook test with execnb on {path}")
    CaptureShell().execute(path, dest=path, exc_stop=False, verbose=False)
    _print_nb_outputs(path)
    errors = _notebook_error_summaries(path)
    if errors:
        sys.stdout.flush()
        _cli_error("Notebook test failed after writing/execution: " + "; ".join(errors))

## Diffing notebooks

The diff command focuses on code-cell source changes, which is the part that matters most when reviewing nbdev notebooks.

In [0]:
#| export
def code_source(cell): return cell.source if cell.cell_type == "code" else None

In [ ]:
#| export
@call_parse
@_tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str|None = "HEAD",  # First git ref; use None for working tree
    ref_b: str|None = None,  # Second git ref; defaults to working tree
    adds: bool = True,  # Include code cells added in ref_b
    changes: bool = True,  # Include changed code cells
    dels: bool = False,  # Include deleted code cells
):
    "Print nbdev-style diffs for code cells only."
    ref_a, ref_b = _none_if_string(ref_a), _none_if_string(ref_b)
    old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    blocks = []
    if adds:    blocks += [(cid, source_diff("", new[cid])) for cid in new if cid not in old]
    if changes: blocks += [(cid, source_diff(old[cid], new[cid])) for cid in new if cid in old and new[cid] != old[cid]]
    if dels:    blocks += [(cid, source_diff(old[cid], "")) for cid in old if cid not in new]
    text = "\n\n".join(f"--- code cell {cid} ---\n{diff}" for cid, diff in blocks if diff.strip())
    if text: print(text)
    else: print("No code cell changes")
    return _cli_return(text)

In [0]:
#| export
def _node_source(lines, node):
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1
    return "\n".join(lines[start:node.end_lineno]).strip("\n")

In [0]:
#| export
def _node_line_count(node):
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]])
    return node.end_lineno - start + 1

In [0]:
#| export
def _patchable_method(node):
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return False
    if node.decorator_list: return False
    return bool(node.args.posonlyargs or node.args.args)

In [0]:
#| export
def _annotate_first_arg(node, class_name):
    args = node.args.posonlyargs or node.args.args
    if not args: return
    args[0].annotation = ast.Name(id=class_name, ctx=ast.Load())

In [0]:
#| export
def _patch_method_source(method, class_name):
    node = copy.deepcopy(method)
    node.decorator_list = []
    _annotate_first_arg(node, class_name)
    ast.fix_missing_locations(node)
    return f"@patch\n{ast.unparse(node)}"

In [0]:
#| export
def _class_without_methods(lines, cls, methods):
    class_start = min([cls.lineno, *[d.lineno for d in cls.decorator_list]])
    class_line = cls.lineno - class_start
    src_lines = lines[class_start - 1:cls.end_lineno]
    remove_ranges = []
    for method in methods:
        start = min([method.lineno, *[d.lineno for d in method.decorator_list]]) - class_start
        stop = method.end_lineno - class_start + 1
        remove_ranges.append((start, stop))
    for start, stop in sorted(remove_ranges, reverse=True): del src_lines[start:stop]
    body = src_lines[class_line + 1:]
    if not any(line.strip() for line in body): src_lines.append("    pass")
    return "\n".join(src_lines).strip("\n")

In [0]:
#| export
def _export_cell(source):
    return mk_cell(f"#| export\n{source.strip()}")

## Converting Python into notebooks

The conversion helpers turn ordinary Python modules into nbdev notebooks so existing code can move into a notebook-first workflow.

In [0]:
#| export
def _py2nb_cells(source, default_exp, class_lines=100, method_lines=10):
    tree = ast.parse(source)
    lines = source.splitlines()
    cells = [mk_cell(f"#| default_exp {default_exp}")]
    pending_imports = []
    needs_patch = False

    def flush_imports():
        if pending_imports:
            cells.append(_export_cell("\n".join(pending_imports)))
            pending_imports.clear()

    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            pending_imports.append(_node_source(lines, node))
            continue
        if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == "__all__" for target in node.targets):
            continue
        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id == "__all__":
            continue
        if isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name) and node.target.id == "__all__":
            continue
        flush_imports()
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): cells.append(_export_cell(_node_source(lines, node)))
        elif isinstance(node, ast.ClassDef):
            methods = [child for child in node.body if _patchable_method(child) and _node_line_count(child) > method_lines]
            if _node_line_count(node) > class_lines and methods:
                needs_patch = True
                cells.append(_export_cell(_class_without_methods(lines, node, methods)))
                for method in methods: cells.append(_export_cell(_patch_method_source(method, node.name)))
            else: cells.append(_export_cell(_node_source(lines, node)))
        elif not (isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str)):
            cells.append(_export_cell(_node_source(lines, node)))
    flush_imports()
    if needs_patch: cells.insert(1, _export_cell("from fastcore.basics import patch"))
    return cells

In [0]:
#| export
def _py2nb_file(path, nbs_path="nbs", dest=None, class_lines=100, method_lines=10):
    pth = Path(path)
    source = pth.read_text(encoding="utf-8")
    default_exp = pth.stem
    out_path = Path(dest) if dest else Path(nbs_path) / f"{default_exp}.ipynb"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nb = new_nb(_py2nb_cells(source, default_exp, class_lines=class_lines, method_lines=method_lines))
    _write_nb(nb, out_path)
    return out_path, len(nb.cells)

In [ ]:
#| export
@call_parse
@_tracked_call
def py2nb(
    path: str,  # Python file path
    nbs_path: str = "nbs",  # Folder for generated notebooks
    dest: str | None = None,  # Explicit notebook path; overrides nbs_path
    class_lines: int = 100,  # Split methods out of classes longer than this
    method_lines: int = 10,  # Split methods longer than this out of large classes
):
    "Convert a Python file into an nbdev-style notebook using AST parsing."
    out_path, n_cells = _py2nb_file(path, nbs_path=nbs_path, dest=dest, class_lines=class_lines, method_lines=method_lines)
    msg = f"Wrote {n_cells} cells to {out_path}"
    print(msg)
    return _cli_return(out_path)

In [ ]:
#| export
@call_parse
@_tracked_call
def py2nbs(
    path: str,  # Folder containing Python files
    nbs_path: str = "nbs",  # Folder for generated notebooks
    recursive: bool = True,  # Search subfolders
    maxdepth: int | None = None,  # Maximum folder depth to search
    preserve_tree: bool = True,  # Preserve folder structure below nbs_path
    class_lines: int = 100,  # Split methods out of classes longer than this
    method_lines: int = 10,  # Split methods longer than this out of large classes
):
    "Convert all Python files in a folder into nbdev-style notebooks."
    root = Path(path)
    py_files = pglob(root, exts="py", recursive=recursive, maxdepth=maxdepth)
    outs = []
    for pth in py_files:
        dest = None
        if preserve_tree and root.is_dir():
            dest = Path(nbs_path) / pth.relative_to(root).with_suffix(".ipynb")
        out_path, n_cells = _py2nb_file(str(pth), nbs_path=nbs_path, dest=str(dest) if dest else None,
                                        class_lines=class_lines, method_lines=method_lines)
        outs.append(out_path)
        print(f"Wrote {n_cells} cells to {out_path}")
    print(f"Converted {len(outs)} Python files to {nbs_path}")
    return _cli_return(outs)

In [0]:
#| export
def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not line.lstrip().startswith("#|"))

In [0]:
#| export
def _annotation_name(annotation):
    if annotation is None: return None
    if isinstance(annotation, ast.Name): return annotation.id
    if isinstance(annotation, ast.Attribute): return annotation.attr
    if isinstance(annotation, ast.Constant): return annotation.value
    return ast.unparse(annotation)

In [0]:
#| export
def _first_arg_annotation(node):
    args = node.args.posonlyargs or node.args.args
    return _annotation_name(args[0].annotation) if args else None

In [0]:
#| export
def _node_defines_symbol(node, symbol):
    parts = symbol.split(".")
    name = parts[-1]
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
        return True
    if len(parts) < 2: return False

    cls_name, meth_name = parts[-2], parts[-1]
    if isinstance(node, ast.ClassDef) and node.name == cls_name:
        return any(isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == meth_name for child in node.body)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == meth_name:
        return _first_arg_annotation(node) == cls_name
    return False

In [0]:
#| export
def _cell_defines_symbol(cell, symbol):
    if cell.cell_type != "code": return False
    try: tree = ast.parse(_source_without_directives(cell.source))
    except SyntaxError: return False
    return any(_node_defines_symbol(node, symbol) for node in tree.body)

In [0]:
#| export
def _find_symbol_cell(nb, symbol):
    for idx, cell in enumerate(nb.cells):
        if _cell_defines_symbol(cell, symbol): return idx
    raise ValueError(f"Could not find symbol {symbol!r}")

In [0]:
#| export
def _find_symbol_node(cell, symbol):
    if getattr(cell, "cell_type", None) != "code": return None
    try: tree = ast.parse(_source_without_directives(cell.source))
    except SyntaxError: return None
    parts = symbol.split(".")
    for node in tree.body:
        if len(parts) == 1 and isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == symbol:
            return node
        if _node_defines_symbol(node, symbol):
            if isinstance(node, ast.ClassDef) and len(parts) > 1:
                name = parts[-1]
                return next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == name), node)
            return node
    return None

In [0]:
#| export
def _previous_markdown(cells, idx, limit):
    docs = []
    pos = idx - 1
    while pos >= 0 and len(docs) < limit:
        cell = cells[pos]
        if getattr(cell, "cell_type", None) != "markdown": break
        docs.append((pos, cell))
        pos -= 1
    return list(reversed(docs))

In [0]:
#| export
def _following_examples(cells, idx, limit):
    examples = []
    pos = idx + 1
    while pos < len(cells) and len(examples) < limit:
        cell = cells[pos]
        if _is_exported_code_cell(cell): break
        if getattr(cell, "cell_type", None) in {"markdown", "code"}: examples.append((pos, cell))
        pos += 1
    return examples

In [0]:
#| export
def _symbol_signature_text(cell, symbol):
    node = _find_symbol_node(cell, symbol)
    if node is None: return _code_overview(cell)
    if isinstance(node, ast.ClassDef): return _class_overview(node)
    return _function_overview(node)

In [ ]:
#| export
def _format_symbol_doc(nb, symbol, context=2, source=False, show_ids=False):
    idx = _find_symbol_cell(nb, symbol)
    cell = nb.cells[idx]
    lines = [f"Symbol {symbol}", "Full context: rationale/docs -> exported code -> show-off examples", _cell_prefix(idx, cell, show_ids)]
    docs = _previous_markdown(nb.cells, idx, context)
    if docs:
        lines.append("")
        lines.append("Rationale/docs before the symbol:")
        for doc_idx, doc_cell in docs:
            lines.append(_cell_prefix(doc_idx, doc_cell, show_ids))
            lines.append(doc_cell.source.strip())
    signature = _symbol_signature_text(cell, symbol)
    if signature:
        lines.append("")
        lines.append("Exported definition:")
        lines.extend(signature)
    if source:
        lines.append("")
        lines.append("Source cell:")
        lines.append(cell.source.strip())
    examples = _following_examples(nb.cells, idx, context)
    if examples:
        lines.append("")
        lines.append("Show-off examples after the symbol:")
        for ex_idx, ex_cell in examples:
            lines.append(_cell_prefix(ex_idx, ex_cell, show_ids))
            lines.append(ex_cell.source.strip())
    return "\n".join(lines)

In [ ]:
#| export
@call_parse
@_tracked_call
def show_doc(
    path: str,  # Notebook path
    symbol: str,  # Function, class, or Class.method to inspect
    context: int = 2,  # Rationale/docs before and show-off example cells after to include
    source: bool = False,  # Include the full source cell
    show_ids: bool = False,  # Include source hashes in output
):
    "Show rationale/docs, exported code, and show-off examples for a notebook symbol."
    nb = _read_nb(path)
    text = _format_symbol_doc(nb, symbol, context=context, source=source, show_ids=show_ids)
    print(text)
    return _cli_return(text)

## Agent workflow helpers

These commands install the notebook skill and add documentation or examples near Python symbols, so agents can work in notebooks instead of generated files.

In [ ]:
#| export
@call_parse
@_tracked_call
def doc4symbol(
    path: str,  # Notebook path
    symbol: str,  # Python symbol, e.g. function, Class, or Class.method
    text: Param("Markdown documentation", str, opt=False),  # Markdown text to insert
    export: bool = True,  # Run nbdev-export after writing
):
    "Insert markdown documentation before a symbol cell."
    path = Path(path)
    nb = _read_nb(path)
    idx = _find_symbol_cell(nb, symbol)
    anchor_id = getattr(nb.cells[idx], "id", "?")
    nb.cells.insert(idx, mk_cell(text, cell_type="markdown"))
    _save_nb(nb, path, export=export)
    msg = f"Inserted documentation for {symbol} before id={anchor_id}"
    print(msg)
    return _cli_return(path)

In [ ]:
#| export
@call_parse
@_tracked_call
def example4symbol(
    path: str,  # Notebook path
    symbol: str,  # Python symbol, e.g. function, Class, or Class.method
    example: Param("Example or test source", str, opt=False),  # Example/test content to insert
    cell_type: str = "code",  # Cell type for the example
    export: bool = True,  # Run nbdev-export after writing
):
    "Insert an example or test cell after a symbol cell."
    path = Path(path)
    nb = _read_nb(path)
    idx = _find_symbol_cell(nb, symbol)
    anchor_id = getattr(nb.cells[idx], "id", "?")
    nb.cells.insert(idx + 1, mk_cell(example, cell_type=cell_type))
    _save_nb(nb, path, export=export)
    msg = f"Inserted example for {symbol} after id={anchor_id}"
    print(msg)
    return _cli_return(path)

In [ ]:
#| export
@call_parse
@_tracked_call
def install_nbskill(
    target: str = "codex",  # codex, claude, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Overwrite an existing SKILL.md
):
    "Install the bundled SKILL.md into a Codex or Claude Code skills directory."
    target = target.lower()
    if skills_dir: roots = [Path(skills_dir).expanduser()]
    elif target == "codex": roots = [Path.home() / ".codex" / "skills"]
    elif target in {"claude", "claude-code", "claude_code"}: roots = [Path.home() / ".claude" / "skills"]
    elif target == "both": roots = [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    else: raise ValueError("target must be codex, claude, both, or use skills_dir")
    skill_text = files("nbskill").joinpath("SKILL.md").read_text(encoding="utf-8")
    installed = []
    for root in roots:
        dst_dir = root / skill_name
        dst = dst_dir / "SKILL.md"
        if dst.exists() and not overwrite: raise FileExistsError(dst)
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst.write_text(skill_text, encoding="utf-8")
        installed.append(dst)
    msg = "\n".join(f"Installed {path}" for path in installed)
    print(msg)
    return _cli_return(installed)

In [ ]:
#| export
@call_parse
@_tracked_call
def update_cell(
    path: str,  # Notebook path
    new: Param("Replacement cell source, or replacement text when old_str is set", str, opt=False, nargs="?") = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    cell_id: str | None = None,  # Stable notebook cell id to update
    old_str: str | None = None,  # Text to replace, or text used to find the target cell
    source_hash: str | None = None,  # Expected current source SHA256 prefix
    cell_type: str = "code",  # Default type for whole-cell replacements without %% marker
    export: bool = True,  # Run nbdev-export after writing
    run_test: bool = False,  # Execute the notebook with execnb after writing
    validate_code: bool = True,  # Validate changed Python code before writing
    dry_run: bool = False,  # Show the update plan without writing
):
    "Update one notebook cell by id, or replace old_str inside one uniquely matching cell."
    if cell_id is None and old_str is None: _cli_error("Pass --cell_id, --old_str, or both")
    path = Path(path)
    nb = _read_nb(path)
    new = _load_cells_text(new, new_file)

    idx, cell = _find_cell_by_id(nb.cells, cell_id) if cell_id else _find_cell_by_text(nb.cells, old_str)
    if old_str is not None and old_str not in _cell_source(cell):
        _cli_error(f"old_str was not found in id={cell.id}")
    if not _cell_matches_hash(cell, source_hash):
        actual = _cell_hash(cell, n=None)
        _cli_error(f"Hash mismatch for id={cell.id}: expected {source_hash}, actual {actual[:12]}")

    before_hash = _cell_hash(cell)
    if old_str is None:
        new_cell = _parse_one_cell(new, cell_type)
        if validate_code: _validate_code_cells([new_cell])
        _clear_outputs(new_cell)
        if not dry_run: _replace_cell(nb, idx, new_cell)
        after_hash = _cell_hash(new_cell)
        mode = "cell"
    else:
        replacement = _cell_source(cell).replace(old_str, new, 1)
        if validate_code and getattr(cell, "cell_type", None) == "code": _validate_code_cells([mk_cell(replacement)])
        after_hash = _cell_hash(replacement)
        mode = "text"
        if not dry_run:
            cell.source = replacement
            _clear_outputs(cell)

    msg = f"{'Dry run: would update' if dry_run else 'Updated'} {mode} id={cell.id} hash={before_hash}->{after_hash}"
    if dry_run:
        print(msg)
        return _cli_return(path)
    _write_nb(nb, path)
    if export: nbdev_export(path=str(path))
    if export: msg += " and exported with nbdev"
    print(msg)
    if run_test: _run_notebook_test(path)
    return _cli_return(path)

## Notebook tests

These cells are intentionally not exported. `nbdev-test` executes them as behavioral checks for the CLI helpers in this notebook.

In [ ]:
import json as _json
import os as _os
import tempfile as _tempfile
from contextlib import redirect_stderr as _redirect_stderr
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from pathlib import Path as _TestPath

_failure_map_dir = _tempfile.TemporaryDirectory()
_failure_map_path = _TestPath(_failure_map_dir.name) / ".nbskill-errors.json"
_os.environ["NBSKILL_FAILURE_MAP"] = str(_failure_map_path)

from nbskill.core import (
    _cell_hash as _cell_hash_cli,
    _output_text as _output_text_cli,
    read_nb as _read_nb_cli,
    show_doc as _show_doc_cli,
    update_cell as _update_cell_cli,
    write_nb as _write_nb_cli,
)


def _quiet_call(func, *args, **kwargs):
    out = _StringIO()
    with _redirect_stdout(out), _redirect_stderr(out):
        result = func(*args, **kwargs)
    return out.getvalue() if result is None else result

In [ ]:
with _tempfile.TemporaryDirectory() as td:
    path = _TestPath(td) / "sample.ipynb"
    _quiet_call(
        _write_nb_cli,
        str(path),
        "%%markdown\n# Title\n---\n%%code\nvalue = 1",
        replace=True,
        export=False,
    )
    code_cell = _read_nb(path).cells[1]

    text = _quiet_call(_read_nb_cli, str(path), scope="full", show_ids=True)
    md_text = _quiet_call(_read_nb_cli, str(path), scope="full", cell_type="md")
    overview = _quiet_call(_read_nb_cli, str(path))

    assert f"id={code_cell.id}" in text
    assert f"hash={_cell_hash_cli(code_cell)}" in text
    assert "# Title" in md_text and "value = 1" not in md_text
    assert "Cell [" not in text
    assert "Cell id=" in overview

In [ ]:
with _tempfile.TemporaryDirectory() as td:
    path = _TestPath(td) / "sample.ipynb"
    _quiet_call(_write_nb_cli, str(path), "%%code\nvalue = 1\nprint(value)", replace=True, export=False)
    cell = _read_nb(path).cells[0]

    _quiet_call(
        _update_cell_cli,
        str(path),
        "value = 2\nprint(value)",
        cell_id=cell.id,
        source_hash=_cell_hash_cli(cell),
        export=False,
    )
    updated = _read_nb(path).cells[0]

    assert updated.id == cell.id
    assert "value = 2" in updated.source

In [ ]:
with _tempfile.TemporaryDirectory() as td:
    path = _TestPath(td) / "sample.ipynb"
    _quiet_call(_write_nb_cli, str(path), "%%markdown\nold text", replace=True, export=False)

    _quiet_call(_update_cell_cli, str(path), "new text", old_str="old text", export=False)

    assert _read_nb(path).cells[0].source == "new text"

In [ ]:
with _tempfile.TemporaryDirectory() as td:
    path = _TestPath(td) / "sample.ipynb"
    _quiet_call(
        _write_nb_cli,
        str(path),
        "%%code\nprint('sandbox-safe test')\nassert 1 + 1 == 2",
        replace=True,
        export=False,
        run_test=True,
    )

    outputs = _read_nb(path).cells[0].outputs
    assert outputs
    assert "sandbox-safe test" in _output_text_cli(outputs[0])

In [ ]:
with _tempfile.TemporaryDirectory() as td:
    path = _TestPath(td) / "sample.ipynb"
    _quiet_call(
        _write_nb_cli,
        str(path),
        "%%markdown\n## Math\nAdds numbers.\n---\n%%code\n#| export\ndef add(a, b):\n    \"\"\"Add two values.\"\"\"\n    return a + b\n---\n%%code\nassert add(1, 2) == 3",
        replace=True,
        export=False,
    )

    text = _quiet_call(_show_doc_cli, str(path), "add", context=1, source=True, show_ids=True)

    assert "Full context: rationale/docs -> exported code -> show-off examples" in text
    assert "Rationale/docs before the symbol:" in text
    assert "Exported definition:" in text
    assert "Show-off examples after the symbol:" in text
    assert "Adds numbers." in text
    assert "def add(a, b):" in text
    assert "Add two values." in text
    assert "assert add(1, 2) == 3" in text
    assert "hash=" in text

In [ ]:
with _tempfile.TemporaryDirectory() as td:
    path = _TestPath(td) / "sample.ipynb"
    _quiet_call(_write_nb_cli, str(path), "%%code\nvalue = 1", replace=True, export=False)

    _quiet_call(_read_nb_cli, str(path))
    _quiet_call(_read_nb_cli, str(path))
    try:
        _quiet_call(_read_nb_cli, str(path), cell_id="missing")
    except (ValueError, SystemExit):
        pass

    data = _json.loads(_failure_map_path.read_text(encoding="utf-8"))
    assert any(event["kind"] == "friction" and event["tool"] == "read_nb" for event in data["events"])
    assert data["counts"]["failures"]["read_nb"] >= 1